## Method 2: using food codes prefix

### Mixed dishes

In [1]:
import pandas as pd

file_path = '../processed_data/foods_list.csv'
foods_df = pd.read_csv(file_path)

wweia_desc_filters = [
    'Meat mixed dishes',
    'Poultry mixed dishes',
    'Seafood mixed dishes',
    'Bean, pea, legume dishes',
    'Vegetable dishes',
    'Rice mixed dishes',
    'Pasta mixed dishes, excludes macaroni & cheese',
    'Macaroni and cheese',
    'Turnovers and other grain-based items',
    'Fried rice and lo/chow mein',
    'Stir-fry and soy-based sauce mixtures',
    'Egg rolls, dumplings, sushi',
    'Burritos and tacos',
    'Nachos',
    'Other Mexican mixed dishes',
    'Pizza',
    'Burgers',
    'Frankfurter sandwiches',
    'Chicken fillet sandwiches',
    'Egg/breakfast sandwiches',
    'Cheese sandwiches',
    'Peanut butter and jelly sandwiches',
    'Seafood sandwiches',
    'Deli and cured meat sandwiches',
    'Meat and BBQ sandwiches',
    'Vegetable sandwiches/burgers',
    'Soups, broth-based',
    'Soups, cream-based',
    'Ramen and Asian broth-based soups'
]

filtered_foods_df = foods_df[foods_df['WWEIA_desc'].isin(wweia_desc_filters)]

num_rows = len(filtered_foods_df)

num_unique_food_desc = filtered_foods_df['food_desc'].nunique()

print(f"Number of rows: {num_rows}")
print(f"Number of unique food descriptions: {num_unique_food_desc}")

Number of rows: 1597
Number of unique food descriptions: 1443


In [2]:
output_file_path = '../processed_data/mixed_dishes.csv'
filtered_foods_df.to_csv(output_file_path, index=False)

### using cosine similarity to remove near-similar foods (the lower the threshold, the fewer foods we will have in the final list)

#### similarity threshold = 0.8

In [13]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

mixed_dishes_df = pd.read_csv('../processed_data/mixed_dishes.csv')
food_tagging_df = pd.read_csv('../processed_data/food_tagging.csv')

food_descriptions = mixed_dishes_df['food_desc'].values

# create a TF-IDF Vectorizer and compute cosine similarity between all food descriptions
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(food_descriptions)
cosine_similarities = cosine_similarity(tfidf_matrix)

# threshold for similarity: define foods with cosine similarity above a certain threshold as "similar"
similarity_threshold = 0.8

# Group similar foods together based on cosine similarity
groups = []
grouped_indices = set()

for i in range(len(cosine_similarities)):
    if i in grouped_indices:
        continue
    
    group = [i]
    grouped_indices.add(i)
    
    for j in range(i+1, len(cosine_similarities)):
        if cosine_similarities[i, j] > similarity_threshold:
            group.append(j)
            grouped_indices.add(j)
    
    groups.append(group)

# merge with food_tagging_df and select the food with most non-zero nutritional tags
def count_non_zero_nutrition_tags(row):
    # count the number of non-zero values in nutritional columns
    return row[[
        'low_calorie', 'high_calorie', 'low_protein', 'high_protein', 'low_carb', 'high_carb',
        'low_sugar', 'high_sugar', 'low_fiber', 'high_fiber', 'low_saturated_fat', 'high_saturated_fat',
        'low_cholesterol', 'high_cholesterol', 'low_sodium', 'high_sodium', 'low_calcium', 'high_calcium',
        'low_phosphorus', 'high_phosphorus', 'low_potassium', 'high_potassium', 'low_iron', 'high_iron',
        'low_folic_acid', 'high_folic_acid', 'low_vitamin_c', 'high_vitamin_c', 'low_vitamin_d', 'high_vitamin_d',
        'low_vitamin_b12', 'high_vitamin_b12'
    ]].sum()

# pick one food from each group based on the nutritional tag count
selected_foods = []

for group in groups:
    group_food_ids = mixed_dishes_df.iloc[group]['food_id']
    
    # merge with food_tagging_df to get nutritional information for the group
    group_foods_with_tags = pd.merge(
        mixed_dishes_df.iloc[group], 
        food_tagging_df, 
        on='food_id',
        how='inner'
    )
    
    # find the food with the most non-zero nutritional tags
    group_foods_with_tags['non_zero_nutrition_count'] = group_foods_with_tags.apply(count_non_zero_nutrition_tags, axis=1)
    
    # select the food with the most non-zero tags
    selected_food = group_foods_with_tags.sort_values(by='non_zero_nutrition_count', ascending=False).iloc[0]
    selected_foods.append(selected_food)

selected_foods_df = pd.DataFrame(selected_foods)

# remove duplicate food_id and keep only one row per food_id
selected_foods_df = selected_foods_df.drop_duplicates(subset=['food_id'])

selected_foods_df.to_csv('../processed_data/reduced_mixed_dishes.csv', index=False)

print(selected_foods_df.head())


    food_id                                          food_desc  \
1  27111410                         Chili con carne with beans   
0  58106520     Pizza with pepperoni, from frozen, thick crust   
2  58145113            Macaroni or noodles with cheese, canned   
0  58163410                            Spanish rice, fat added   
2  27111300  Beef stew, no potatoes, tomato-based sauce, Me...   

            WWEIA_desc                                    ingredient_desc  \
1    Meat mixed dishes          Chili con carne with beans, canned entree   
0                Pizza  Pizza, cheese topping, rising crust, frozen, c...   
2  Macaroni and cheese                 Macaroni and Cheese, canned entree   
0    Rice mixed dishes                               Salt, table, iodized   
2    Meat mixed dishes                                          Oil, corn   

      calorie    protein       carb     sugar     fiber  saturated_fat  ...  \
1  102.514829   7.640270   9.288902  2.495260  2.653248      

#### similarity threshold = 0.7

In [14]:
mixed_dishes_df = pd.read_csv('../processed_data/mixed_dishes.csv')
food_tagging_df = pd.read_csv('../processed_data/food_tagging.csv')

food_descriptions = mixed_dishes_df['food_desc'].values

# create a TF-IDF Vectorizer and compute cosine similarity between all food descriptions
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(food_descriptions)
cosine_similarities = cosine_similarity(tfidf_matrix)

# threshold for similarity: define foods with cosine similarity above a certain threshold as "similar"
similarity_threshold = 0.7

# Group similar foods together based on cosine similarity
groups = []
grouped_indices = set()

for i in range(len(cosine_similarities)):
    if i in grouped_indices:
        continue
    
    group = [i]
    grouped_indices.add(i)
    
    for j in range(i+1, len(cosine_similarities)):
        if cosine_similarities[i, j] > similarity_threshold:
            group.append(j)
            grouped_indices.add(j)
    
    groups.append(group)

# merge with food_tagging_df and select the food with most non-zero nutritional tags
def count_non_zero_nutrition_tags(row):
    # count the number of non-zero values in nutritional columns
    return row[[
        'low_calorie', 'high_calorie', 'low_protein', 'high_protein', 'low_carb', 'high_carb',
        'low_sugar', 'high_sugar', 'low_fiber', 'high_fiber', 'low_saturated_fat', 'high_saturated_fat',
        'low_cholesterol', 'high_cholesterol', 'low_sodium', 'high_sodium', 'low_calcium', 'high_calcium',
        'low_phosphorus', 'high_phosphorus', 'low_potassium', 'high_potassium', 'low_iron', 'high_iron',
        'low_folic_acid', 'high_folic_acid', 'low_vitamin_c', 'high_vitamin_c', 'low_vitamin_d', 'high_vitamin_d',
        'low_vitamin_b12', 'high_vitamin_b12'
    ]].sum()

# pick one food from each group based on the nutritional tag count
selected_foods = []

for group in groups:
    group_food_ids = mixed_dishes_df.iloc[group]['food_id']
    
    # merge with food_tagging_df to get nutritional information for the group
    group_foods_with_tags = pd.merge(
        mixed_dishes_df.iloc[group], 
        food_tagging_df, 
        on='food_id',
        how='inner'
    )
    
    # find the food with the most non-zero nutritional tags
    group_foods_with_tags['non_zero_nutrition_count'] = group_foods_with_tags.apply(count_non_zero_nutrition_tags, axis=1)
    
    # select the food with the most non-zero tags
    selected_food = group_foods_with_tags.sort_values(by='non_zero_nutrition_count', ascending=False).iloc[0]
    selected_foods.append(selected_food)

selected_foods_df = pd.DataFrame(selected_foods)

# remove duplicate food_id and keep only one row per food_id
selected_foods_df = selected_foods_df.drop_duplicates(subset=['food_id'])

selected_foods_df.to_csv('../processed_data/reduced_mixed_dishes.csv', index=False)

print(selected_foods_df.head())


    food_id                                          food_desc  \
1  27111410                         Chili con carne with beans   
2  58106200             Pizza, cheese, from frozen, thin crust   
3  58145113            Macaroni or noodles with cheese, canned   
2  58163430                         Spanish rice, NS as to fat   
2  27111300  Beef stew, no potatoes, tomato-based sauce, Me...   

            WWEIA_desc                                    ingredient_desc  \
1    Meat mixed dishes          Chili con carne with beans, canned entree   
2                Pizza  Pizza, cheese topping, thin crust, frozen, cooked   
3  Macaroni and cheese                 Macaroni and Cheese, canned entree   
2    Rice mixed dishes                                 Vegetable oil, NFS   
2    Meat mixed dishes                                          Oil, corn   

      calorie    protein       carb     sugar     fiber  saturated_fat  ...  \
1  102.514829   7.640270   9.288902  2.495260  2.653248      

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def test_cosine_similarity(text1, text2):
    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform([text1, text2])

    cosine_sim = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])

    return cosine_sim[0][0]

# Example usage:
# text1 = "Beef, noodles, and vegetables including carrots, broccoli, and/or dark-green leafy; mushroom sauce"
# text2 = "Beef, potatoes, and vegetables excluding carrots, broccoli, and dark-green leafy; tomato-based sauce"

# text1 = 'Egg burrito, with bacon'
# text2 = 'Egg burrito, with ham'

text1 = 'Rice, brown, with gravy, fat added'
text2 = 'Rice, brown, with gravy, NS as to fat'

similarity_score = test_cosine_similarity(text1, text2)
print(f"Cosine similarity score between the two texts: {similarity_score}")

Cosine similarity score between the two texts: 0.5727393584196199


#### cosine similarity threshold = 0.6

In [15]:
mixed_dishes_df = pd.read_csv('../processed_data/mixed_dishes.csv')
food_tagging_df = pd.read_csv('../processed_data/food_tagging.csv')

food_descriptions = mixed_dishes_df['food_desc'].values

# create a TF-IDF Vectorizer and compute cosine similarity between all food descriptions
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(food_descriptions)
cosine_similarities = cosine_similarity(tfidf_matrix)

# threshold for similarity: define foods with cosine similarity above a certain threshold as "similar"
similarity_threshold = 0.6

# Group similar foods together based on cosine similarity
groups = []
grouped_indices = set()

for i in range(len(cosine_similarities)):
    if i in grouped_indices:
        continue
    
    group = [i]
    grouped_indices.add(i)
    
    for j in range(i+1, len(cosine_similarities)):
        if cosine_similarities[i, j] > similarity_threshold:
            group.append(j)
            grouped_indices.add(j)
    
    groups.append(group)

# merge with food_tagging_df and select the food with most non-zero nutritional tags
def count_non_zero_nutrition_tags(row):
    # count the number of non-zero values in nutritional columns
    return row[[
        'low_calorie', 'high_calorie', 'low_protein', 'high_protein', 'low_carb', 'high_carb',
        'low_sugar', 'high_sugar', 'low_fiber', 'high_fiber', 'low_saturated_fat', 'high_saturated_fat',
        'low_cholesterol', 'high_cholesterol', 'low_sodium', 'high_sodium', 'low_calcium', 'high_calcium',
        'low_phosphorus', 'high_phosphorus', 'low_potassium', 'high_potassium', 'low_iron', 'high_iron',
        'low_folic_acid', 'high_folic_acid', 'low_vitamin_c', 'high_vitamin_c', 'low_vitamin_d', 'high_vitamin_d',
        'low_vitamin_b12', 'high_vitamin_b12'
    ]].sum()

# pick one food from each group based on the nutritional tag count
selected_foods = []

for group in groups:
    group_food_ids = mixed_dishes_df.iloc[group]['food_id']
    
    # merge with food_tagging_df to get nutritional information for the group
    group_foods_with_tags = pd.merge(
        mixed_dishes_df.iloc[group], 
        food_tagging_df, 
        on='food_id',
        how='inner'
    )
    
    # find the food with the most non-zero nutritional tags
    group_foods_with_tags['non_zero_nutrition_count'] = group_foods_with_tags.apply(count_non_zero_nutrition_tags, axis=1)
    
    # select the food with the most non-zero tags
    selected_food = group_foods_with_tags.sort_values(by='non_zero_nutrition_count', ascending=False).iloc[0]
    selected_foods.append(selected_food)

selected_foods_df = pd.DataFrame(selected_foods)

# remove duplicate food_id and keep only one row per food_id
selected_foods_df = selected_foods_df.drop_duplicates(subset=['food_id'])

selected_foods_df.to_csv('../processed_data/reduced_mixed_dishes.csv', index=False)

print(selected_foods_df.head())


    food_id                                          food_desc  \
1  27111410                         Chili con carne with beans   
5  58106200             Pizza, cheese, from frozen, thin crust   
3  58145113            Macaroni or noodles with cheese, canned   
2  58163430                         Spanish rice, NS as to fat   
1  27111310  Beef stew, no potatoes, tomato-based sauce, wi...   

            WWEIA_desc                                    ingredient_desc  \
1    Meat mixed dishes          Chili con carne with beans, canned entree   
5                Pizza  Pizza, cheese topping, thin crust, frozen, cooked   
3  Macaroni and cheese                 Macaroni and Cheese, canned entree   
2    Rice mixed dishes                                 Vegetable oil, NFS   
1    Meat mixed dishes                               Salt, table, iodized   

      calorie    protein       carb     sugar     fiber  saturated_fat  ...  \
1  102.514829   7.640270   9.288902  2.495260  2.653248      

### another method: compare first few words

In [19]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

mixed_dishes_df = pd.read_csv('../processed_data/mixed_dishes.csv')
food_tagging_df = pd.read_csv('../processed_data/food_tagging.csv')

# remove special characters and get the first few words of a food description
def get_first_words(food_desc, num_words=3):
    clean_desc = re.sub(r'[^a-zA-Z0-9\s]', '', food_desc).lower()
    # extract the first 'num_words' words
    return ' '.join(clean_desc.split()[:num_words])

mixed_dishes_df['first_words'] = mixed_dishes_df['food_desc'].apply(lambda x: get_first_words(x, num_words=3))

# group foods by their first few words
grouped_foods = mixed_dishes_df.groupby('first_words')

# merge with food_tagging_df and select the food with the most non-zero nutritional tags
def count_non_zero_nutrition_tags(row):
    # count the number of non-zero values in nutritional columns
    return row[[
        'low_calorie', 'high_calorie', 'low_protein', 'high_protein', 'low_carb', 'high_carb',
        'low_sugar', 'high_sugar', 'low_fiber', 'high_fiber', 'low_saturated_fat', 'high_saturated_fat',
        'low_cholesterol', 'high_cholesterol', 'low_sodium', 'high_sodium', 'low_calcium', 'high_calcium',
        'low_phosphorus', 'high_phosphorus', 'low_potassium', 'high_potassium', 'low_iron', 'high_iron',
        'low_folic_acid', 'high_folic_acid', 'low_vitamin_c', 'high_vitamin_c', 'low_vitamin_d', 'high_vitamin_d',
        'low_vitamin_b12', 'high_vitamin_b12'
    ]].sum()

# pick one food from each group based on the nutritional tag count
selected_foods = []

for group_name, group in grouped_foods:
    group_food_ids = group['food_id']
    
    group_foods_with_tags = pd.merge(
        group, 
        food_tagging_df, 
        on='food_id',
        how='inner'
    )
    
    # find the food with the most non-zero nutritional tags
    group_foods_with_tags['non_zero_nutrition_count'] = group_foods_with_tags.apply(count_non_zero_nutrition_tags, axis=1)
    
    # select the food with the most non-zero tags
    selected_food = group_foods_with_tags.sort_values(by='non_zero_nutrition_count', ascending=False).iloc[0]
    selected_foods.append(selected_food)

selected_foods_df = pd.DataFrame(selected_foods)

selected_foods_df = selected_foods_df.drop_duplicates(subset=['food_id'])

selected_foods_df.to_csv('../processed_data/reduced_mixed_dishes_v4.csv', index=False)

# Function to create reduced foods list for a specified category and list of WWEIA_desc filters

In [16]:
import pandas as pd
import re
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings

warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

def process_foods(category, wweia_desc_filters, num_first_words):
    file_path = '../processed_data/foods_list.csv'
    foods_df = pd.read_csv(file_path)
    
    # filter foods_df for rows where WWEIA_desc is in the specified list
    filtered_foods_df = foods_df[foods_df['WWEIA_desc'].isin(wweia_desc_filters)]
    
    output_file_path = f'../processed_data/{category}.csv'
    filtered_foods_df.to_csv(output_file_path, index=False)
    
    print(f"Number of rows: {len(filtered_foods_df)}")
    print(f"Number of unique food descriptions: {filtered_foods_df['food_desc'].nunique()}")
    
    mixed_dishes_df = pd.read_csv(output_file_path)
    food_tagging_df = pd.read_csv('../processed_data/food_tagging.csv')

    # remove special characters and get the first few words of a food description
    def get_first_words(food_desc, num_words=3):
        clean_desc = re.sub(r'[^a-zA-Z0-9\s]', '', food_desc).lower()
        return ' '.join(clean_desc.split()[:num_words])

    mixed_dishes_df['first_words'] = mixed_dishes_df['food_desc'].apply(lambda x: get_first_words(x, num_words=num_first_words))

    # group foods by their first few words
    grouped_foods = mixed_dishes_df.groupby('first_words')

    # count the number of non-zero nutritional tags in specified columns
    def count_non_zero_nutrition_tags(row, columns):
        return row[columns].sum()

    # primary and secondary sets of nutritional columns
    primary_nutrition_columns = [
        'low_calorie', 'high_calorie', 'low_protein', 'high_protein', 'low_carb', 'high_carb',
        'low_sugar', 'high_sugar', 'low_fiber', 'high_fiber', 'low_saturated_fat', 'high_saturated_fat',
        'low_cholesterol', 'high_cholesterol', 'low_sodium', 'high_sodium', 'low_calcium', 'high_calcium',
        'low_phosphorus', 'high_phosphorus', 'low_potassium', 'high_potassium', 'low_iron', 'high_iron',
        'low_folic_acid', 'high_folic_acid', 'low_vitamin_c', 'high_vitamin_c', 'low_vitamin_d', 'high_vitamin_d',
        'low_vitamin_b12', 'high_vitamin_b12'
    ]

    secondary_nutrition_columns = [
        'low_calorie', 'high_calorie', 'low_protein', 'high_protein', 'low_carb', 'high_carb',
        'low_sugar', 'high_sugar', 'low_fiber', 'high_fiber', 'low_saturated_fat', 'high_saturated_fat',
        'low_cholesterol', 'high_cholesterol', 'low_sodium', 'high_sodium'
    ]

    # pick one food from each group based on the nutritional tag count
    selected_foods = []

    for group_name, group in grouped_foods:
        group_food_ids = group['food_id']
        
        group_foods_with_tags = pd.merge(
            group, 
            food_tagging_df, 
            on='food_id',
            how='inner'
        )
        
        # find the food with the most non-zero nutritional tags (primary set)
        group_foods_with_tags['non_zero_primary_count'] = group_foods_with_tags.apply(count_non_zero_nutrition_tags, axis=1, columns=primary_nutrition_columns)
        
        # get the foods with the highest count in primary nutrition columns
        top_primary_foods = group_foods_with_tags.sort_values(by='non_zero_primary_count', ascending=False)
        top_primary_count = top_primary_foods['non_zero_primary_count'].iloc[0]
        tied_foods = top_primary_foods[top_primary_foods['non_zero_primary_count'] == top_primary_count]
        
        # If there's a tie, compare by the secondary set of nutritional columns
        if len(tied_foods) > 1:
            tied_foods['non_zero_secondary_count'] = tied_foods.apply(count_non_zero_nutrition_tags, axis=1, columns=secondary_nutrition_columns)
            top_secondary_foods = tied_foods.sort_values(by='non_zero_secondary_count', ascending=False)
            top_secondary_count = top_secondary_foods['non_zero_secondary_count'].iloc[0]
            tied_foods_secondary = top_secondary_foods[top_secondary_foods['non_zero_secondary_count'] == top_secondary_count]
            
            # If there's still a tie, randomly choose one
            if len(tied_foods_secondary) > 1:
                selected_food = tied_foods_secondary.sample(1).iloc[0]
            else:
                selected_food = tied_foods_secondary.iloc[0]
        else:
            selected_food = tied_foods.iloc[0]

        selected_foods.append(selected_food)

    selected_foods_df = pd.DataFrame(selected_foods)

    selected_foods_df = selected_foods_df.drop_duplicates(subset=['food_id'])

    selected_foods_df = selected_foods_df.drop_duplicates(subset=['food_desc'])

    selected_foods_df.to_csv(f'../processed_data/reduced_{category}.csv', index=False)
    
    print(f"Processed category '{category}' and saved reduced food list.")


In [2]:
import os
import pandas as pd
import time
from autogen import ConversableAgent
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

llm_config = {
    "model": "gpt-3.5-turbo",
    "api_key": api_key
}

# define the food recommendation agent
agent = ConversableAgent(
    name="food_recommendation_agent",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

# dynamically create a recommendable criteria based on the dataset
def create_recommendable_criteria(instructions):
    return f"""
    The food description must represent a dish that is suitable for recommendation to users. 
    {instructions}
    
    Recommendable foods are those that can be part of a recipe or a full dish.
    """

# determine if a food is recommendable
def is_food_recommendable(food_desc, recommendable_criteria, retries=3):
    prompt = f"""
    You are a food recommendation agent. Your task is to judge whether a food description should be recommended to users or not.
    Follow the criteria below:
    {recommendable_criteria}

    Food Description: "{food_desc}"
    Should this food be recommended? Answer "Yes" or "No" with a brief explanation.
    """
    
    for attempt in range(retries):
        try:
            response = agent.generate_reply(
                messages=[{"content": prompt, "role": "user"}]
            )
            answer = response.lower().strip()
            
            if "yes" in answer:
                return "recommendable", answer
            elif "no" in answer:
                return "non-recommendable", answer
            else:
                return "non-recommendable", "Unclear response: " + answer

        except Exception as e:
            print(f"Attempt {attempt + 1} failed for '{food_desc}' with error: {e}")
            time.sleep(2 * (attempt + 1))

    return "non-recommendable", "Error during evaluation after retries"

# process a specific file and customize the instructions for food items
def process_food_file(category, instructions):
    # create recommendable criteria based on the instructions
    recommendable_criteria = create_recommendable_criteria(instructions)
    
    file_path = f'../processed_data/reduced_{category}.csv'
    foods_df = pd.read_csv(file_path)

    # loop through each food item to determine if it's recommendable
    food_results = []

    for idx, row in foods_df.iterrows():
        food_desc = row['food_desc']
        recommendable_flag, reasoning = is_food_recommendable(food_desc, recommendable_criteria)
        
        food_results.append({
            "food_id": row['food_id'],
            "food_desc": food_desc,
            "WWEIA_desc": row['WWEIA_desc'],
            "ingredient_desc": row.get('ingredient_desc', ''),
            "recommendable_flag": recommendable_flag,
            "reasoning": reasoning
        })
        
        # print(f"Food '{food_desc}': {recommendable_flag.capitalize()} - {reasoning}")

        time.sleep(2)

    food_results_df = pd.DataFrame(food_results)

    output_path = f'../processed_data/reduced_{category}_with_recommendations.csv'
    food_results_df.to_csv(output_path, index=False)
    print(f"Results saved to {output_path}")

flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.


[autogen.oai.client: 10-08 20:33:16] {184} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.


In [9]:
category = "mixed_dishes"
wweia_desc_filters = [
    'Meat mixed dishes',
    'Poultry mixed dishes',
    'Seafood mixed dishes',
    'Bean, pea, legume dishes',
    'Vegetable dishes',
    'Rice mixed dishes',
    'Pasta mixed dishes, excludes macaroni & cheese',
    'Macaroni and cheese',
    'Turnovers and other grain-based items',
    'Fried rice and lo/chow mein',
    'Stir-fry and soy-based sauce mixtures',
    'Egg rolls, dumplings, sushi',
    'Burritos and tacos',
    'Nachos',
    'Other Mexican mixed dishes',
    'Pizza',
    'Burgers',
    'Frankfurter sandwiches',
    'Chicken fillet sandwiches',
    'Egg/breakfast sandwiches',
    'Cheese sandwiches',
    'Peanut butter and jelly sandwiches',
    'Seafood sandwiches',
    'Deli and cured meat sandwiches',
    'Meat and BBQ sandwiches',
    'Vegetable sandwiches/burgers',
    'Soups, broth-based',
    'Soups, cream-based',
    'Ramen and Asian broth-based soups'
]

process_foods(category, wweia_desc_filters, num_first_words=3)

Number of rows: 1597
Number of unique food descriptions: 1443
Processed category 'mixed_dishes' and saved reduced food list.


In [10]:
category = "meat_seafood"
wweia_desc_filters = [
'Beef, excludes ground',
'Ground beef',
'Pork',
'Lamb, goat, game',
'Liver and organ meats',
'Chicken, whole pieces',
'Chicken patties, nuggets and tenders',
'Turkey, duck, other poultry',
'Fish',
'Shellfish',
'Eggs and omelets',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Number of rows: 1176
Number of unique food descriptions: 933
Processed category 'meat_seafood' and saved reduced food list.


In [3]:
category = "meat_seafood"
instructions = """
Avoid foods that: 
1. Are raw
2. Do not include a specific cooking method in their description
"""

process_food_file(category, instructions)

Attempt 1 failed for 'Fish, eel' with error: 'NoneType' object has no attribute 'lower'
Attempt 1 failed for 'Pompano, coated, fried' with error: 'NoneType' object has no attribute 'lower'
Attempt 1 failed for 'Veal, NS as to cut, cooked, NS as to fat eaten' with error: 'NoneType' object has no attribute 'lower'
Results saved to ../processed_data/reduced_meat_seafood_with_recommendations.csv


In [5]:
category = "processed_meat"
wweia_desc_filters = [
'Cold cuts and cured meats',
'Bacon',
'Frankfurters',
'Sausages',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Number of rows: 175
Number of unique food descriptions: 118
Processed category 'processed_meat' and saved reduced food list.


In [ ]:
# category = "processed_meat"
# instructions = """
# Avoid foods that: 
# """

# process_food_file(category, instructions)

In [6]:
category = "plant_protein"
wweia_desc_filters = [
'Beans, peas, legumes',
'Nuts and seeds',
'Soy and meat-alternative products',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Number of rows: 265
Number of unique food descriptions: 243
Processed category 'plant_protein' and saved reduced food list.


In [7]:
category = "plant_protein"
instructions = """
Avoid foods that: 
1. Are raw or unprocessed
2. Do not include a specific cooking method in their description
"""

process_food_file(category, instructions)

Attempt 1 failed for 'Stewed chickpeas, Puerto Rican style' with error: 'NoneType' object has no attribute 'lower'
Results saved to ../processed_data/reduced_plant_protein_with_recommendations.csv


In [17]:
category = "breads"
wweia_desc_filters = [
'Yeast breads',
'Rolls and buns',
'Bagels and English muffins',
'Tortillas',
'Biscuits, muffins, quick breads',
'Pancakes, waffles, French toast',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Number of rows: 354
Number of unique food descriptions: 308
Processed category 'breads' and saved reduced food list.


In [ ]:
# category = "breads"
# instructions = """
# Avoid foods that: 
# """

# process_food_file(category, instructions)

In [8]:
category = "baked_desserts"
wweia_desc_filters = [
'Cakes and pies',
'Cookies and brownies',
'Doughnuts, sweet rolls, pastries',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Number of rows: 525
Number of unique food descriptions: 383
Processed category 'baked_desserts' and saved reduced food list.


In [ ]:
# category = "baked_desserts"
# instructions = """
# Avoid foods that: 
# """

# process_food_file(category, instructions)

In [9]:
category = "vegetables_potatoes"
wweia_desc_filters = [
'Tomatoes',
'Carrots',
'Other red and orange vegetables',
'Broccoli',
'Spinach',
'Lettuce and lettuce salads',
'Other dark green vegetables',
'String beans',
'Cabbage',
'Onions',
'Corn',
'Other starchy vegetables',
'Other vegetables and combinations',
'Fried vegetables',
'Coleslaw, non-lettuce salads',
'Vegetables on a sandwich',
'White potatoes, baked or boiled',
'French fries and other fried white potatoes',
'Mashed potatoes and white potato mixtures',
]

process_foods(category, wweia_desc_filters, num_first_words=2)

Number of rows: 1112
Number of unique food descriptions: 1048
Processed category 'vegetables_potatoes' and saved reduced food list.


In [11]:
category = "vegetables_potatoes"
instructions = """
Avoid foods that: 
1. Are raw vegetables/potatoes or uncooked vegetables/potatoes
2. Do not include a specific cooking method in their description
"""

process_food_file(category, instructions)

Attempt 1 failed for 'Cauliflower, raw' with error: 'NoneType' object has no attribute 'lower'
Attempt 1 failed for 'Peppers, red, cooked, fat not added in cooking' with error: 'NoneType' object has no attribute 'lower'
Results saved to ../processed_data/reduced_vegetables_potatoes_with_recommendations.csv
